# Assignment: week 4

Objectives

The objectives of this assignment are:

1. to learn how to obtain and use pretrained word embeddings
2. to gain a better understanding of word vectors


In [49]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

## Preparing data

### Reading data

Here I created paths for the ``GloVe Wikipedia 2014 + Gigaword 5`` data that I downloaded.

In [50]:
base = "../datasets/GloVe-pretrained"
dims = [50, 100, 200, 300]

glove_paths = {f"{d}d": f"{base}/glove.6B.{d}d.txt" for d in dims}

Next I read the selected file and save it as `data`. I then save the word embeddings into a dictionary. The dictionary entires are later used as my vocabulary with words and weightings.

In [51]:
selected_dimension = "300d"
file_path = glove_paths[selected_dimension]

embeddings = {}
with open(file_path, "r", encoding="utf-8") as data:
    for line in data:
        content = line.strip().split()
        word = content[0]
        vec = np.asarray(content[1:], dtype="float32")
        embeddings[word] = vec

Here I create the vocabulary, indexing and weighting array.

The ``vec()`` method takes in a word as its argument and returns its vector / weights in the shape we selected earlier.
- `selected_dimension = "300d"`
- ``shape(300, 0)``

The ``nearest_words()`` method takes in the vector and returns `n` amount of words most similar by cosine similarity. It also allows for exclusion of words.

In [52]:
vocabulary = list(embeddings.keys())
word_index = {word: idx for idx, word in enumerate(vocabulary)}
index_word = {idx: word for word, idx in word_index.items()}
embedding_weights = np.vstack([embeddings[w] for w in vocabulary])

def vec(word):
    if word not in embeddings:
        raise ValueError(f'Word "{word}" not in vocabulary')
    return embeddings[word]


def nearest_words(vector, n=5, exclude=None):
    if exclude is None:
        exclude = set()

    sims = cosine_similarity(vector.reshape(1, -1), embedding_weights).reshape(-1)
    sorted = np.argsort(sims)[::-1]

    results = []
    for idx in sorted:
        word = index_word[idx]
        if word in exclude:
            continue
        results.append((word))
        if len(results) == n:
            break
    return results

In [ ]:
print(vec("woman").shape) # Shape of the vector

(300,)


In [98]:
result = vec("woman") - vec("man") + vec("king")
top = nearest_words(result, n=5, exclude={"woman", "man", "king"})
print(top)

['queen', 'monarch', 'throne', 'princess', 'mother']


### Summary v.01


The function returns `n = 5` amount of nearest words. The results show pretty convincing results, and something one would roughly expect:

- `['queen', 'monarch', 'throne', 'princess', 'mother']`

with the assignments given expression:

- `vec("woman") - vec("man) + vec("king")`

## Made up combinations

### First combination

First I want to see if the pretrained embeddings can calculate the following combination:

- `laptop - components + software`

to equal:

- `computer / personal computer`

In [54]:
result = vec("laptop") - vec("components") + vec("software")
top = nearest_words(result, n=5, exclude={"laptop", "components", "software"})

print(top)

['computer', 'computers', 'desktop', 'laptops', 'pc']


The results shows top five nearest words and they do indeed match the expected result of `computer`.

### Second combination

Second combination I came up with was:

- `video - virtual + player`

to see if it would equal to:

- `game`

In [91]:
result = vec("video") - vec("virtual") + vec("player")
top = nearest_words(result, 5, {"video", "virtual", "player"})

print(top)

['players', 'playing', 'game', 'played', 'play']


From the results I could see that the values are very tied into the `player` words weighting. This can be seen from the 4/5 words that had the word `play` in them. In the end I did get the result that I wanted, which was `game`!

### Summary v.02

From the couple of tests and some tinkering that I did not include in the homework notebook, I came to the conclusion that the cosine similarity can calculate the expressions **surprisingly** well with the given data and their weightings. 

**Conclusion:**

Most of the time, if my expressions made at least some sense, the function could return somewhat reasonable results.